# Cohort Creation Workflow (Steps 1b → 2)

Final workflow for creating cohorts. One cell per cohort series; run Configuration once, then the cohort cell(s) you need.

## Order of operations

1. **Configuration** — Set project root and paths (run once).
2. **Cohort 1 (FALLS)** — Builds all age-band × event-year partitions for the `fall_injury_any` outcome.
3. **Cohort 2 (ED)** — Builds all age-band × event-year partitions for the `ed_event` outcome.
4. **Check status** — Pipeline state and durations in S3; cohort parquets in S3 data lake. Use `--outputs` for file listing; use `--logs` for build logs.

## Cohorts

| Series | Script | Target |
|--------|--------|--------|
| **FALLS** | `run_series_falls.py` | `fall_injury_any` (injury S00–S99 + external cause W00–W19 for the same patient within `CPIC_FALL_TARGET_WINDOW_DAYS`, default 7 days) |
| **ED** | `run_series_ed.py` | `ed_event` (POS=23 or revenue code 045x/0981) |

## Prerequisites

- **Step 1a** — APCD input data in gold (medical/pharmacy).
- **Step 1b** — Event filter producing `fall_injury_any`, `fall_injury_serious`, `fall_injury_head`, `ed_event` flags plus feature flags (`r29_6_flag`, `z91_81_flag`, `cpt_1100f_flag`).

## Configuration

In [1]:
import sys
from pathlib import Path

# Project root (notebook at repo root)
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PYTHON_BIN = Path(sys.executable)
LOG_DIR = PROJECT_ROOT / "2_create_cohort" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Log dir: {LOG_DIR}")

Project root: /home/pgx3874/cpic_time_to_event_analysis
Python: /home/pgx3874/jupyter-env/bin/python3.11
Log dir: /home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/logs


## Cohort 1: FALLS

Runs the FALLS series for all age bands and event years. Each partition writes `fall_injury_any` cohort parquets to S3. Partitions that already have the cohort output are skipped (`--skip-existing`).

In [7]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_falls.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

Running: /home/pgx3874/jupyter-env/bin/python3.11 /home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/run_series_falls.py --skip-existing --concurrent-workers 1
2026-06-20 12:15:25,687 - INFO - ================================================================================
2026-06-20 12:15:25,688 - INFO - 🚀 OPTIMIZED COHORT CREATION PIPELINE
2026-06-20 12:15:25,688 - INFO - ================================================================================
2026-06-20 12:15:25,688 - INFO - 📊 Age Band: 65-74
2026-06-20 12:15:25,688 - INFO - 📊 Event Year: 2016
2026-06-20 12:15:25,688 - INFO - 📊 Cohort Type: falls
2026-06-20 12:15:25,688 - INFO - 📊 Starting Step: phase1_data_preparation
2026-06-20 12:15:25,688 - INFO - 📊 Operation Type: concurrent_processing
2026-06-20 12:15:25,688 - INFO - 📊 Profiling: Disabled
2026-06-20 12:15:25,688 - INFO - 📊 Process ID: 53280
2026-06-20 12:15:25,688 - INFO - 📊 CPU Cores Available: 32
2026-06-20 12:15:25,688 - INFO - 📊 Concurrent Workers Detected: 

CompletedProcess(args=['/home/pgx3874/jupyter-env/bin/python3.11', '/home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/run_series_falls.py', '--skip-existing', '--concurrent-workers', '1'], returncode=0)

## Cohort 2: ED

Runs the ED series for all age bands and event years. Partitions that already have the cohort output are skipped (`--skip-existing`). Use this cell to run ED-only or to force-rebuild (omit `--skip-existing` if needed).

In [2]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_ed.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

Running: /home/pgx3874/jupyter-env/bin/python3.11 /home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/run_series_ed.py --skip-existing --concurrent-workers 1
2026-06-20 17:19:15,069 - INFO - ================================================================================
2026-06-20 17:19:15,069 - INFO - [START] OPTIMIZED COHORT CREATION PIPELINE
2026-06-20 17:19:15,069 - INFO - ================================================================================
2026-06-20 17:19:15,069 - INFO - [INFO] Age Band: 65-74
2026-06-20 17:19:15,069 - INFO - [INFO] Event Year: 2018
2026-06-20 17:19:15,069 - INFO - [INFO] Cohort Type: ed
2026-06-20 17:19:15,069 - INFO - [INFO] Starting Step: phase1_data_preparation
2026-06-20 17:19:15,069 - INFO - [INFO] Operation Type: concurrent_processing
2026-06-20 17:19:15,069 - INFO - [INFO] Profiling: Disabled
2026-06-20 17:19:15,069 - INFO - [INFO] Process ID: 19968
2026-06-20 17:19:15,069 - INFO - [INFO] CPU Cores Available: 32
2026-06-20 17:19:15,069 

CompletedProcess(args=['/home/pgx3874/jupyter-env/bin/python3.11', '/home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/run_series_ed.py', '--skip-existing', '--concurrent-workers', '1'], returncode=0)

## Check S3 completion status

Lists pipeline state (pgx-repository) with status and **duration** per partition, and cohort parquet counts in pgxdatalake. Add `--outputs` to the script args to list each parquet with LastModified and size; add `--logs` to list build logs from pgx-repository.

In [3]:
import subprocess
script = PROJECT_ROOT / "2_create_cohort" / "check_s3_cohort_completion.py"
# Use --outputs to list each cohort.parquet with LastModified and size
subprocess.run([str(PYTHON_BIN), str(script), "--outputs"], cwd=str(PROJECT_ROOT))

COHORT PIPELINE STATE (production + legacy fallback)
Production: s3://pgxdatalake/gold/cpic_time_to_event/pipeline_checkpoints/create_cohort/
Legacy:     s3://pgx-repository/pgx-pipeline-status/create_cohort/
ENTITY_ID                 STATUS      CREATED           COMPLETED         DURATION  
------------------------------------------------------------------------------------
ed_65-74_2016             completed   2026-06-20 16:11  2026-06-20 16:29  17.7m     
ed_65-74_2017             completed   2026-06-20 16:29  2026-06-20 16:46  17.7m     
ed_65-74_2018             completed   2026-06-20 16:46  2026-06-20 17:43  56.4m     
ed_65-74_2019             completed   2026-06-20 17:43  2026-06-20 18:04  21.1m     
ed_75-84_2016             completed   2026-06-20 18:04  2026-06-20 18:16  11.8m     
ed_75-84_2017             completed   2026-06-20 18:16  2026-06-20 18:27  11.8m     
ed_75-84_2018             completed   2026-06-20 18:28  2026-06-20 18:43  15.1m     
ed_75-84_2019             

CompletedProcess(args=['/home/pgx3874/jupyter-env/bin/python3.11', '/home/pgx3874/cpic_time_to_event_analysis/2_create_cohort/check_s3_cohort_completion.py', '--outputs'], returncode=0)